# Azure AI Agent와 MCP


## 1. 환경 설정

필요한 라이브러리를 가져오고 환경 변수를 로드합니다.

In [1]:
import os
import json
import time
import requests
import subprocess
from dotenv import load_dotenv

load_dotenv("../../env")

endpoint = os.getenv("END_POINT", "").rstrip("/")
api_key  = os.getenv("AZURE_OPENAI_API_KEY")
model    = os.getenv("MODEL_NAME")

# Azure OpenAI 엔드포인트 자동 보정
if ".openai.azure.com" in endpoint and "/openai/deployments/" not in endpoint:
    endpoint = f"{endpoint}/openai/deployments/{model}"

print(f"Target Endpoint: {endpoint}")
print(f"Model Name     : {model}")

Target Endpoint: https://kt-new-foundry-resource.openai.azure.com/openai/deployments/gpt-5-nano
Model Name     : gpt-5-nano


## 2. MCP 서버 실행 (Background)


In [2]:
!fuser -k 8000/tcp

server_process = subprocess.Popen(["python", "server_mcp.py"])

time.sleep(5)

8000/tcp:            19850


INFO:     Started server process [70156]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


## 3. MCP 클라이언트 구현

우리의 Azure Agent가 이 MCP 서버와 통신할 수 있도록 **Bridge 함수**를 만듭니다.     
이 함수는 에이전트가 `get_weather` 도구를 호출하려고 할 때, 실제로 로컬호스트의 8000번 포트로 요청을 보냅니다.

In [3]:
def call_mcp_server(method, params=None):
    url = "http://localhost:8000/mcp"
    payload = {
        "jsonrpc": "2.0",
        "method": method,
        "params": params or {},
        "id": 1
    }
    response = requests.post(url, json=payload)
    return response.json()

tools_info = call_mcp_server("tools/list")
print("MCP에서 사용할 수 있는 Tools:")
print(json.dumps(tools_info, indent=2, ensure_ascii=False))

INFO:     127.0.0.1:56482 - "POST /mcp HTTP/1.1" 200 OK
MCP에서 사용할 수 있는 Tools:
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "tools": [
      {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "inputSchema": {
          "type": "object",
          "properties": {
            "city": {
              "type": "string",
              "description": "The name of the city"
            }
          },
          "required": [
            "city"
          ]
        }
      }
    ]
  }
}


## 4. Azure AI Agent 정의

### 주요 클래스

1.  **메시지 클래스 (Messages)**
    -   SystemMessage: AI의 역할과 성격을 부여. (예: "너는 친절한 날씨 알리미야")
    -   UserMessage: 사용자가 입력한 질문이나 요청.
    -   AssistantMessage: AI가 생성한 응답. (텍스트 답변 또는 도구 호출 요청)
    -   ToolMessage: 도구(함수)가 실행된 후 그 결과값을 담는 메시지.

2.  **도구 정의 클래스 (Tool Definitions)**
    -   ChatCompletionsToolDefinition: 모델에게 "이런 도구가 있어"라고 알려줌.
    -   FunctionDefinition: 실제 도구의 **이름**, **설명**, **파라미터(입력값)** 정보.

### 동적 도구 생성
**FunctionDefinition**을 일일이 코딩하지 않고, 앞서 **MCP 서버에서 받아온 정보(`tools/list`)**를 반복문으로 돌면서,    
Azure SDK가 이해할 수 있는 **ChatCompletionsToolDefinition** 객체로 변환할 것입니다.  


In [4]:
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    AssistantMessage,
    ToolMessage,
    ChatCompletionsToolDefinition,
    FunctionDefinition,
)
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

credential = AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()

client = ChatCompletionsClient(endpoint=endpoint, credential=credential)

mcp_tools = tools_info.get("result", {}).get("tools", [])
azure_tools = []

for tool in mcp_tools:
    azure_tools.append(
        ChatCompletionsToolDefinition(
            function=FunctionDefinition(
                name=tool["name"],
                description=tool["description"],
                parameters=tool["inputSchema"]
            )
        )
    )

print(f"Loaded {len(azure_tools)} tools into Azure Agent.")

Loaded 1 tools into Azure Agent.


## 5. 에이전트 실행 (Conversation Loop)

사용자와 대화하며 필요할 때 MCP 서버를 호출하는 루프를 실행.

1. 사용자 질문 입력
2. 모델이 Tool 호출 결정
3. **MCP 서버로 요청 전달 (`call_mcp_server`)**
4. 결과를 모델에 반환하여 최종 답변 생성

In [5]:
def run_agent(user_query):
    messages = [
        SystemMessage(content="당신은 유용한 어시스턴트입니다. 제공된 도구를 사용해 사용자의 질문에 답하세요."),
        UserMessage(content=user_query)
    ]
    
    print(f"\nUser: {user_query}")
    
    response = client.complete(
        messages=messages,
        tools=azure_tools,
        tool_choice="auto"
    )
    
    assistant_msg = response.choices[0].message
    print(f"\nAssistant: {assistant_msg.tool_calls}")
    
    if assistant_msg.tool_calls: #<-- 도구 호출이 있는지 확인
        messages.append(AssistantMessage(content=assistant_msg.content, tool_calls=assistant_msg.tool_calls))
        
        for tool_call in assistant_msg.tool_calls:
            print(f"Calling Tool: {tool_call.function.name}...")
            
            # MCP 서버로 실제 요청 전송
            args = json.loads(tool_call.function.arguments)
            mcp_response = call_mcp_server(
                method="tools/call",
                params={"name": tool_call.function.name, "arguments": args}
            )
            
            # 결과 추출
            tool_result = mcp_response["result"]["content"][0]["text"]
            print(f"   - Tool Result: {tool_result}")
            
            messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call.id))

        final_response = client.complete(messages=messages)
        print(f"Agent: {final_response.choices[0].message.content}")
        
    else:
        print(f"Agent: {assistant_msg.content}")

# 테스트 실행
run_agent("안녕?") #<-- 이때는 tool을 agent가 호출하지 않습니다.
print('--------------------------------------------')
run_agent("서울의 날씨는 어때?")
print('--------------------------------------------')
run_agent("뉴욕 날씨도 알려줘")


User: 안녕?

Assistant: None
Agent: 안녕하세요! 무엇을 도와드릴까요?

예를 들어 요청 가능 항목들:
- 특정 도시의 현재 날씨 확인
- 간단한 정보 검색이나 번역
- 요리 레시피나 팁 찾기
- 계산이나 일정 관리 등

날씨를 원하시면 도시 이름을 알려주시면 바로 현재 날씨를 알려드릴게요. 어떤 걸 도와드릴까요?
--------------------------------------------

User: 서울의 날씨는 어때?

Assistant: [{'function': {'arguments': '{"city":"Seoul"}', 'name': 'get_weather'}, 'id': 'call_AiVvrqVj07MTuOjqdlePGl1X', 'type': 'function'}]
Calling Tool: get_weather...
INFO:     127.0.0.1:39694 - "POST /mcp HTTP/1.1" 200 OK
   - Tool Result: Sunny, 25°C
Agent: 서울의 현재 날씨는 맑고 화창하며 기온은 약 25°C입니다. 외출하기 좋은 날씨예요. 필요하시면 시간대별 예보나 주간 예보도 확인해 드릴게요. 자외선이 강할 수 있으니 선크림이나 선글라스를 준비하는 것도 좋습니다.
--------------------------------------------

User: 뉴욕 날씨도 알려줘

Assistant: [{'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'id': 'call_0EbhQK8YV14ltdJeDYOzXURF', 'type': 'function'}]
Calling Tool: get_weather...
INFO:     127.0.0.1:56040 - "POST /mcp HTTP/1.1" 200 OK
   - Tool Result: Cloudy, 18°C
Agent: 뉴욕의 현재 날씨는 흐림이고 기온은 18°C입니